# Notebook 04: Forecasting, scenarios, and conclusions

Derived from **`mmr comprehensive.ipynb`**. This notebook **re-trains the best model on full history**, builds **forward projections** (trend-based global forecast and **scenario** branches: baseline, health investment, climate stress, development), visualizes **2030** comparisons against an illustrative SDG line, and runs **fixed-effects panel OLS** (`linearmodels`) as in the original notebook.

**Prerequisites:** Notebooks 01–03 (same `features` and `best_model` logic). This notebook reloads data and **re-selects** `best_model` by the same validation loop so it can run on its own.

**Data:** `../data/processed/country_year_modeling_panel_cleaned.csv`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance, PartialDependenceDisplay


In [ ]:
import pandas as pd
df = pd.read_csv("../data/processed/country_year_modeling_panel_cleaned.csv")
df["year"] = df["year"].astype(int)
df.head()


In [ ]:
# Ensure year is int and iso3c clean
df["year"] = df["year"].astype(int)
df["iso3c"] = df["iso3c"].astype(str).str.strip().str.upper()

#because of the skewness, applied a log transformation to stabilize variance and improve model performance
#log_mmr = log(1 + MMR)
# If log_mmr isn't in the file, create it
if "log_mmr" not in df.columns:
    df["log_mmr"] = np.log1p(df["mmr"])

target = "log_mmr"

features = [
    "gdp_pc",
    "health_exp_gdp",
    "fertility",
    "skilled_births",
    "pm25",
    "heat_index35_days",
    "cooling_degree_days",
]

# Keep only features that exist
features = [c for c in features if c in df.columns]
print("Using features:", features)


In [ ]:
#included in model: gdp per capita, health expenditure, fertility rate, skilled birth attendance, pm2.5 exposure(pollution), extreme heat dates, and cooling degree days
#variables represent economic, health systems, demographic, and climate drivers on MM
features = [
    "gdp_pc",
    "health_exp_gdp",
    "fertility",
    "skilled_births",
    "pm25",
    "heat_index35_days",
    "cooling_degree_days"
]

features = [f for f in features if f in df.columns]


In [ ]:
#to prevent future data leakage and mimic real-world forecasting
#Training data: Years ≤ 2018
#Test data: Years > 2018

split_year = 2018

train = df[df["year"] <= split_year].copy()
test  = df[df["year"] > split_year].copy()

X_train = train[features]
y_train = train[target]

X_test  = test[features]
y_test  = test[target]

print("Train:", X_train.shape, " Test:", X_test.shape)


In [ ]:
numeric_transformer_scaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

numeric_transformer_unscaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])


In [ ]:
models = {
    #linear:
    "Linear Regression": Pipeline([
        ("prep", numeric_transformer_scaled),
        ("model", LinearRegression())
    ]),
    "Ridge": Pipeline([
        ("prep", numeric_transformer_scaled),
        ("model", Ridge(alpha=1.0))
    ]),
    "Lasso": Pipeline([
        ("prep", numeric_transformer_scaled),
        ("model", Lasso(alpha=0.01))
    ]),
    #kernel based
    "SVR (RBF)": Pipeline([
        ("prep", numeric_transformer_scaled),
        ("model", SVR(kernel="rbf", C=10, epsilon=0.1))
    ]),
    #distance based
    "KNN": Pipeline([
        ("prep", numeric_transformer_scaled),
        ("model", KNeighborsRegressor(n_neighbors=10))
    ]),
    #neural network
    "MLP Regressor": Pipeline([
        ("prep", numeric_transformer_scaled),
        ("model", MLPRegressor(hidden_layer_sizes=(64,32), max_iter=1000, random_state=42))
    ]),
    #tree based
    "Random Forest": Pipeline([
        ("prep", numeric_transformer_unscaled), 
        ("model", RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1))
    ]),
    "Gradient Boosting": Pipeline([
        ("prep", numeric_transformer_unscaled),
        ("model", GradientBoostingRegressor(random_state=42))
    ]),
}


In [ ]:
# evaluate models using MAE, RMSE, R2
results = []

for name, model in models.items():

    model.fit(X_train, y_train)
    preds_log = model.predict(X_test)

    preds = np.expm1(preds_log)
    true = np.expm1(y_test)

    mae = mean_absolute_error(true, preds)
    rmse = np.sqrt(mean_squared_error(true, preds))
    r2 = r2_score(y_test, preds_log)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 (log-scale)": r2
    })

results_df = pd.DataFrame(results).sort_values("RMSE")
results_df


In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = models[best_name]
best_model.fit(X_train, y_train)

print("Best model:", best_name)


## Trend-based global forecast through 2030

(Re-fits `best_model` on all years, then projects predictors per country.)


In [ ]:
#forecasted global average MMR (2024–2030)
#using trend-projected predictors, so the model forecasts future global average maternal mortality rates

#train on full historical data
X_all = df[features]
y_all = df["log_mmr"]
best_model.fit(X_all, y_all)

#latest row per country
latest = df.sort_values("year").groupby("iso3c").tail(1).copy()

future_years = list(range(df["year"].max()+1, 2031))
future = pd.concat([latest.assign(year=y) for y in future_years], ignore_index=True)

#trend-project predictors per country
def project_feature(df_hist, col, horizon_years):
    df_hist = df_hist.sort_values("year")
    if df_hist[col].notna().sum() < 3:
        return np.repeat(df_hist[col].iloc[-1], len(horizon_years))

    x = df_hist["year"].values
    y = df_hist[col].values
    slope, intercept = np.polyfit(x, y, 1)
    return intercept + slope*np.array(horizon_years)

hist = df.copy()
projected = []

for iso, g in hist.groupby("iso3c"):
    yrs = future_years
    row = {"iso3c": iso}
    for col in features:
        row[col] = project_feature(g, col, yrs)
    tmp = pd.DataFrame(row)
    tmp["year"] = yrs
    projected.append(tmp)

proj_df = pd.concat(projected, ignore_index=True)

#predict
proj_df["pred_log_mmr"] = best_model.predict(proj_df[features])
proj_df["pred_mmr"] = np.expm1(proj_df["pred_log_mmr"])

global_forecast = proj_df.groupby("year")["pred_mmr"].mean().reset_index()

plt.figure(figsize=(9,5))
plt.plot(global_forecast["year"], global_forecast["pred_mmr"])
plt.title("Forecasted Global Average MMR (Trend-projected predictors)")
plt.xlabel("Year")
plt.ylabel("Predicted MMR")
plt.show()


## Scenario construction (baseline, health, climate, development)

From the comprehensive notebook: simple extrapolation from `latest` cross-section.


In [ ]:
future_years = list(range(2024, 2031))

future = []

for year in future_years:

    temp = latest.copy()

    years_ahead = year - 2023

    # apply gradual trends
    temp["year"] = year
    temp["gdp_pc"] = temp["gdp_pc"] * (1.03 ** years_ahead)
    temp["health_exp_gdp"] = temp["health_exp_gdp"] * (1.02 ** years_ahead)
    temp["fertility"] = temp["fertility"] * (0.985 ** years_ahead)

    future.append(temp)

future = pd.concat(future, ignore_index=True)


In [ ]:
future_baseline = future.copy()


In [ ]:
#Scenario 1: health investment
#assume +20% health spending and +10% skilled birth attendance

future_health = future.copy()

future_health["health_exp_gdp"] *= 1.25
future_health["skilled_births"] *= 1.10


In [ ]:
#scenario 2: climate stress
#assume PM2.5 increases 15% and heat index days increase 20%
future_climate = future.copy()

future_climate["pm25"] *= 1.20
future_climate["heat_index35_days"] *= 1.25


In [ ]:
#scenario 3- development 
#assume GDP per capita grows and fertility declines
future_dev = future.copy()

future_dev["gdp_pc"] *= 1.30
future_dev["fertility"] *= 0.85


In [ ]:
for df_s in [future_baseline, future_health, future_climate, future_dev]:

    df_s["pred_log_mmr"] = best_model.predict(df_s[features])
    df_s["pred_mmr"] = np.expm1(df_s["pred_log_mmr"])


In [ ]:
#global forecast visualization
baseline_global = future_baseline.groupby("year")["pred_mmr"].mean()
health_global = future_health.groupby("year")["pred_mmr"].mean()
climate_global = future_climate.groupby("year")["pred_mmr"].mean()
dev_global = future_dev.groupby("year")["pred_mmr"].mean()

plt.figure(figsize=(9,6))

plt.plot(baseline_global, label="Baseline")
plt.plot(health_global, label="Health Investment")
plt.plot(climate_global, label="Climate Stress")
plt.plot(dev_global, label="Development Progress")

plt.axhline(70, linestyle="--", label="SDG Target (70)")

plt.ylabel("Global Avg MMR")
plt.xlabel("Year")
plt.title("Forecasted Global Maternal Mortality (2030 Scenarios)")
plt.legend()
plt.show()


In [ ]:
final_year = 2030

vals = {
"Baseline": baseline_global.loc[final_year],
"Health Investment": health_global.loc[final_year],
"Climate Stress": climate_global.loc[final_year],
"Development Progress": dev_global.loc[final_year]
}

plt.figure(figsize=(7,5))
plt.bar(vals.keys(), vals.values())

plt.axhline(70, linestyle="--", color="red")
plt.ylabel("Global Avg MMR (2030)")
plt.title("2030 Maternal Mortality by Scenario")
plt.show()


## Fixed-effects panel model (`linearmodels`)

Country fixed effects with clustered SEs (as in the original notebook).


In [ ]:
#causality analysis
from linearmodels.panel import PanelOLS
import statsmodels.api as sm

panel=df.set_index(["iso3c","year"])
y=panel["log_mmr"]
X = panel[[
    "gdp_pc",
    "health_exp_gdp",
    "fertility",
    "skilled_births",
    "pm25",
    "heat_index35_days"
]]

X = sm.add_constant(X)


In [ ]:
model = PanelOLS(
    y,
    X,
    entity_effects=True
)

results = model.fit(cov_type="clustered", cluster_entity=True)

print(results.summary)


## Conclusions (edit for your report)

Summarize how **scenario assumptions** affect projected global MMR by **2030**, how they compare to the **SDG 3.1** reference line, and how **panel fixed-effects** results align or contrast with **predictive** model findings. Discuss **limitations** (extrapolation of predictors, aggregate global means, causal interpretation).

---
*Replace this section with your own synthesis and policy implications.*
